# AVE Dataset Explorer (MokA)

This notebook mirrors MokA's AVE loading path for a few samples, renders question/audio/video/output, and adds a small `M_q` vs full-text cross-attention ablation demo.

## 1) Environment Setup and Imports

This section resolves the project paths, imports the same libraries used by MokA data loading, and verifies that AVE metadata files are available.

Why this matters:
- Keeps notebook behavior aligned with the training code path.
- Fails early if the notebook is launched from the wrong directory.

In [6]:
from pathlib import Path
import json
import numpy as np
import torch

from IPython.display import display, Markdown, Audio, Video

# Optional imports used in feature extraction (same path as unified_dataset.py)
import librosa
from PIL import Image
from decord import VideoReader
from transformers import CLIPImageProcessor

# MokA audio fbank preprocessing
import sys

# Resolve project root robustly for both execution locations:
# - MokA/AudioVisualText
# - MokA/AudioVisualText/notebooks
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if (REPO_ROOT / 'dataset').exists():
    AVT_ROOT = REPO_ROOT
elif (REPO_ROOT / 'AudioVisualText' / 'dataset').exists():
    AVT_ROOT = REPO_ROOT / 'AudioVisualText'
else:
    raise RuntimeError('Run this notebook from MokA or MokA/AudioVisualText.')

# Add AVT root to Python path so we can import local dataset utilities
sys.path.insert(0, str(AVT_ROOT))
from dataset.audio_processor import preprocess

# Canonical AVE paths used by MokA
AVE_ROOT = AVT_ROOT / 'AVE_data'
TRAIN_JSON = AVE_ROOT / 'train_samples_ave.json'
TEST_JSON = AVE_ROOT / 'test_samples_ave.json'

# Candidate CLIP processor paths (cluster + local fallback)
CLIP_PATH_CANDIDATES = [
    Path('/coc/flash5/rkhan96/weights/clip-vit-large-patch14'),
    Path('/nethome/rkhan96/flash/weights/clip-vit-large-patch14'),
    Path('clip-vit-large-patch14'),
]

display(Markdown(f'**AVT root:** `{AVT_ROOT}`  \n**AVE root:** `{AVE_ROOT}`'))
print('train json exists:', TRAIN_JSON.exists(), 'test json exists:', TEST_JSON.exists())


**AVT root:** `/coc/flash5/rkhan96/MokA/AudioVisualText`  
**AVE root:** `/coc/flash5/rkhan96/MokA/AudioVisualText/AVE_data`

train json exists: True test json exists: True


## 2) Code Provenance: Where MokA Loads and Uses AVE Data

This section prints exact line ranges from the repository so the notebook output is traceable back to source code.

We inspect:
- AVE sample construction and modal loading in `unified_dataset.py`
- Question-mask construction in `unified_arch.py`
- Question-only cross-attention in `peft_hyper/tuners/lora.py`

In [7]:
# Show exact MokA code regions for provenance
def show_snippet(path, start, end):
    # Print source with line numbers so references are easy to verify
    lines = path.read_text().splitlines()
    for i in range(start, end + 1):
        if 1 <= i <= len(lines):
            print(f'{i:4d}: {lines[i-1]}')

unified_dataset_py = AVT_ROOT / 'dataset' / 'unified_dataset.py'
unified_arch_py = AVT_ROOT / 'models' / 'unified_arch.py'
lora_py = AVT_ROOT / 'peft_hyper' / 'tuners' / 'lora.py'

print('--- unified_dataset.py :: AVE sample construction ---')
show_snippet(unified_dataset_py, 90, 114)

print('\n--- unified_dataset.py :: AVE __getitem__ video/audio loading ---')
show_snippet(unified_dataset_py, 197, 239)

print('\n--- unified_arch.py :: question mask construction ---')
show_snippet(unified_arch_py, 145, 170)

print('\n--- lora.py :: question-only cross attention ---')
show_snippet(lora_py, 390, 447)


--- unified_dataset.py :: AVE sample construction ---
  90:     def add_ave_task_samples(self):
  91:         ave_annotation_path = 'AVE_data/train_samples_ave.json'
  92:         ave_data_root = 'AVE_data'
  93:         tot = 0
  94:         with open(ave_annotation_path,'r') as f:
  95:             samples = json.load(f)
  96:         for sample in samples:
  97:             event = sample['event']
  98:             vid = sample['vid']
  99:             start_time = sample['start_time']
 100:             end_time = sample['end_time']
 101:             audio_path = join(ave_data_root,'audio_data',vid+'.mp3')
 102:             video_path = join(ave_data_root,'AVE',vid+'.mp4')
 103:             label_path = join(ave_data_root,'converted_label',vid+'.txt')
 104:             output = self.read_label(label_path)
 105:             instruction = f'This is a video:\n<video_start><video><video_end>\nThis is an audio:\n<audio_start><audio><audio_end>\n<question_start>Please describe the events 

## 3) Build AVE Samples Exactly Like MokA

This section reconstructs AVE sample metadata in the same format MokA uses during training:
- fixed instruction template (question text)
- derived audio/video paths
- output loaded from `converted_label/{vid}.txt`

In [8]:
# Load a few AVE metadata samples and reconstruct instruction/output exactly like unified_dataset.py
def read_label(vid):
    # In AVE, training target text is stored per sample id in converted_label
    label_path = AVE_ROOT / 'converted_label' / f'{vid}.txt'
    return label_path.read_text().strip() if label_path.exists() else '<missing label>'

with open(TRAIN_JSON, 'r') as f:
    train_samples = json.load(f)

# This prompt string is copied from UnifiedDataset.add_ave_task_samples
INSTRUCTION_AVE = (
    'This is a video:\n<video_start><video><video_end>\n'
    'This is an audio:\n<audio_start><audio><audio_end>\n'
    '<question_start>Please describe the events and time range that occurred in the video.<question_end>'
)

# Select a few samples for manual inspection
N = 3
probe = []
for s in train_samples[:N]:
    vid = s['vid']
    probe.append({
        'vid': vid,
        'event': s['event'],
        'start_time': s['start_time'],
        'end_time': s['end_time'],
        'video_path': str(AVE_ROOT / 'AVE' / f'{vid}.mp4'),
        'audio_path': str(AVE_ROOT / 'audio_data' / f'{vid}.mp3'),
        'instruction': INSTRUCTION_AVE,
        'output': read_label(vid),
    })

# Print concise summary table
for i, s in enumerate(probe):
    print(f'[{i}] vid={s["vid"]} event={s["event"]} gt=({s["start_time"]},{s["end_time"]})')
    print('   video:', s['video_path'])
    print('   audio:', s['audio_path'])
    print('   output:', s['output'])


[0] vid=c---zaDCTaE event=Church bell gt=(0,10)
   video: /coc/flash5/rkhan96/MokA/AudioVisualText/AVE_data/AVE/c---zaDCTaE.mp4
   audio: /coc/flash5/rkhan96/MokA/AudioVisualText/AVE_data/audio_data/c---zaDCTaE.mp3
   output: The video shows a church bell tower. The bells in the tower are ringing from the beginning of the video to the end, that is, from the 0th second to the 10th second. So the audible and visible event in the video is <event> Church bell </event>, and the time range is <range> 0,10 </range>.
[1] vid=fCZi6I6kPpU event=Church bell gt=(6,10)
   video: /coc/flash5/rkhan96/MokA/AudioVisualText/AVE_data/AVE/fCZi6I6kPpU.mp4
   audio: /coc/flash5/rkhan96/MokA/AudioVisualText/AVE_data/audio_data/fCZi6I6kPpU.mp3
   output: The video shows a church bell tower. From the beginning of the video until the 6th second, there is no sound. The church bells ring from the 6th second to the 10th second. So the audible and visible event in the video is <event> Church bell </event>, and the 

## 4) Render a Sample (Question + Video + Audio + Label)

This section displays one sample in a human-readable format so you can quickly inspect the instruction text, supervision label, and raw media.

In [9]:
# Render/link a sample's text, audio, video, output
def show_sample(idx=0, width=480):
    s = probe[idx]
    display(Markdown(f'## Sample {idx} - `{s["vid"]}`'))

    # Show instruction/question text used as model input
    display(Markdown(f'**Instruction / question text**\n\n```\n{s["instruction"]}\n```'))

    # Show expected output text from converted label
    display(Markdown(f'**Target output**\n\n```\n{s["output"]}\n```'))

    vpath = Path(s['video_path'])
    apath = Path(s['audio_path'])
    print('Video exists:', vpath.exists(), vpath)
    print('Audio exists:', apath.exists(), apath)

    # Use notebook-native renderers (links + embedded controls)
    if vpath.exists():
        display(Video(filename=str(vpath), width=width, embed=False))
    if apath.exists():
        display(Audio(filename=str(apath)))

show_sample(0)


## Sample 0 - `c---zaDCTaE`

**Instruction / question text**

```
This is a video:
<video_start><video><video_end>
This is an audio:
<audio_start><audio><audio_end>
<question_start>Please describe the events and time range that occurred in the video.<question_end>
```

**Target output**

```
The video shows a church bell tower. The bells in the tower are ringing from the beginning of the video to the end, that is, from the 0th second to the 10th second. So the audible and visible event in the video is <event> Church bell </event>, and the time range is <range> 0,10 </range>.
```

Video exists: True /coc/flash5/rkhan96/MokA/AudioVisualText/AVE_data/AVE/c---zaDCTaE.mp4
Audio exists: True /coc/flash5/rkhan96/MokA/AudioVisualText/AVE_data/audio_data/c---zaDCTaE.mp3


## 5) Reproduce AVE Modal Feature Loading

This section mirrors `UnifiedDataset.__getitem__(task_name == "ave")`:
- video: uniform temporal frame sampling + CLIP preprocessing
- audio: split into 10 one-second chunks + fbank extraction via `audio_processor.preprocess`

The printed shapes help verify what is fed into MokA encoders.

In [10]:
# AVE modal feature extraction mirrored from UnifiedDataset.__getitem__ (task_name == 'ave')
def resolve_clip_path():
    # Pick first existing CLIP checkpoint directory from candidates
    for p in CLIP_PATH_CANDIDATES:
        if p.exists():
            return str(p)
    raise FileNotFoundError('Could not find clip-vit-large-patch14 in known locations.')

clip_path = resolve_clip_path()
video_processor = CLIPImageProcessor.from_pretrained(clip_path)

def load_ave_modal_features(sample, image_size=224, video_frame_nums=10):
    video_path = sample['video_path']
    audio_path = sample['audio_path']

    # ---------------------------
    # Video branch (uniform sampling)
    # ---------------------------
    vr = VideoReader(uri=video_path, height=image_size, width=image_size)
    vlen = len(vr)
    n_frms = min(video_frame_nums, vlen)
    indices = np.arange(0, vlen, vlen / n_frms).astype(int).tolist()
    temp_frms = vr.get_batch(indices).asnumpy()  # T,H,W,C
    frames = [Image.fromarray(temp_frms[i]) for i in range(temp_frms.shape[0])]
    video = video_processor.preprocess(frames, return_tensors='pt')['pixel_values']  # T,C,H,W

    # ---------------------------
    # Audio branch (10 x 1s chunks for AVE)
    # ---------------------------
    audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    length = len(audio)
    tot = 10
    nums_per_second = int(length / tot)
    audio_feature = []

    for indice in range(tot):
        start_time = max(0, indice)
        end_time = min(tot, indice + 1)
        audio_seg = audio[int(start_time * nums_per_second): int(nums_per_second * end_time)]

        # Right-pad with silence if segment is short
        if len(audio_seg) < 1 * nums_per_second:
            sil = np.zeros(1 * nums_per_second - len(audio_seg), dtype=float)
            audio_seg = np.concatenate((audio_seg, sil), axis=0)

        # Same fbank path as training
        audio_seg = torch.from_numpy(audio_seg).unsqueeze(0)
        fbank = preprocess(audio_seg).squeeze(0).to(torch.float32)  # L,128
        audio_feature.append(fbank)

    audio_feature = torch.stack(audio_feature, dim=0)  # T,L,128

    return {'video': video, 'audio': audio_feature, 'frame_indices': indices}

# Run for first two probe samples and print resulting tensor shapes
for i in range(min(2, len(probe))):
    modal = load_ave_modal_features(probe[i])
    print(f'Sample {i} ({probe[i]["vid"]})')
    print('  video shape:', tuple(modal['video'].shape), 'frame idx:', modal['frame_indices'])
    print('  audio shape:', tuple(modal['audio'].shape))


Sample 0 (c---zaDCTaE)
  video shape: (10, 3, 224, 224) frame idx: [0, 25, 50, 75, 100, 125, 150, 175, 200, 225]
  audio shape: (10, 98, 128)
Sample 1 (fCZi6I6kPpU)
  video shape: (10, 3, 224, 224) frame idx: [0, 25, 50, 75, 100, 125, 150, 175, 200, 225]
  audio shape: (10, 98, 128)


## 6) Ablation Demo: `M_q` (Question-Only) vs Full-Text Attention

This reproduces the logic used in `peft_hyper/tuners/lora.py` at a toy level:
- video/audio tokens are queries
- text LoRA branch tokens are keys/values
- compare using only question-span keys (`M_q`) versus all text tokens

The printed norms and attention mass provide a quick sanity check of behavioral differences.

In [11]:
# M_q (question-only) vs full-text ablation demo based on peft_hyper/tuners/lora.py
torch.manual_seed(7)

def cross_update(video_token, audio_token, text_token, video_mask, audio_mask, key_mask, d_k=4, blc_weight=1.0):
    # Video update: query=video_token, key/value=text_token filtered by key_mask
    qv = video_token
    kv = text_token * key_mask
    idx = torch.where(key_mask.squeeze(-1) == 1)[0]
    kv = kv[idx[0]:idx[-1]+1, :]
    score_v = (qv @ kv.T) / (d_k ** 0.5)
    attn_v = torch.softmax(score_v, dim=-1)
    out_v = attn_v @ kv
    new_video = video_token + (video_mask * out_v) * blc_weight

    # Audio update: same structure as video update
    qa = audio_token
    score_a = (qa @ kv.T) / (d_k ** 0.5)
    attn_a = torch.softmax(score_a, dim=-1)
    out_a = attn_a @ kv
    new_audio = audio_token + (audio_mask * out_a) * blc_weight

    return new_video, new_audio, attn_v, attn_a

# Toy sequence in LoRA A-space (d_k=4), with many non-question text tokens as distractors
T_text = 20
T_video = 8
T_audio = 8
d_k = 4

text_token = torch.randn(T_text, d_k)
video_token = torch.randn(T_video, d_k)
audio_token = torch.randn(T_audio, d_k)

# Mask variants
question_mask = torch.zeros(T_text, 1)
question_mask[14:20] = 1  # last 6 text tokens are question span
full_text_mask = torch.ones(T_text, 1)
video_mask = torch.ones(T_video, d_k)
audio_mask = torch.ones(T_audio, d_k)

# Run both settings
nv_q, na_q, attv_q, atta_q = cross_update(video_token, audio_token, text_token, video_mask, audio_mask, question_mask, d_k=d_k)
nv_f, na_f, attv_f, atta_f = cross_update(video_token, audio_token, text_token, video_mask, audio_mask, full_text_mask, d_k=d_k)

def avg_question_mass(attn, q_start=14, q_end=20):
    # If only question tokens are present in KV, mass is exactly 1.0
    if attn.shape[1] == (q_end - q_start):
        return 1.0
    return attn[:, q_start:q_end].sum(dim=-1).mean().item()

print('Video delta norm (Mq):      ', (nv_q - video_token).norm().item())
print('Video delta norm (full txt):', (nv_f - video_token).norm().item())
print('Audio delta norm (Mq):      ', (na_q - audio_token).norm().item())
print('Audio delta norm (full txt):', (na_f - audio_token).norm().item())
print('Avg question-attn mass (video, Mq):      ', avg_question_mass(attv_q))
print('Avg question-attn mass (video, full txt):', avg_question_mass(attv_f))
print('Avg question-attn mass (audio, Mq):      ', avg_question_mass(atta_q))
print('Avg question-attn mass (audio, full txt):', avg_question_mass(atta_f))


Video delta norm (Mq):       2.1200497150421143
Video delta norm (full txt): 3.0253419876098633
Audio delta norm (Mq):       2.3169591426849365
Audio delta norm (full txt): 2.559093475341797
Avg question-attn mass (video, Mq):       1.0
Avg question-attn mass (video, full txt): 0.23980757594108582
Avg question-attn mass (audio, Mq):       1.0
Avg question-attn mass (audio, full txt): 0.29861217737197876


## Notes
- AVE question text is fixed in training/test JSON path (`unified_dataset.py`):
  `Please describe the events and time range that occurred in the video.`
- `M_q` in MokA is built in `models/unified_arch.py` and consumed in `peft_hyper/tuners/lora.py` to restrict key/value to question-region text.
- The ablation above compares this question-only behavior to using all text tokens as keys/values.
